#### Huggingface diffusers library

In [ ]:
pip install diffusers matplotlib 
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

### Installations

In [ ]:
!nvidia-smi

In [ ]:
%pip install diffusers

In [ ]:
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from diffusers import DDPMScheduler, UNet2DModel
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### Pre-process dataset

In [ ]:
import os

root_dir = "train"  # or your dataset root

for class_name in sorted(os.listdir(root_dir)):
    class_path = os.path.join(root_dir, class_name)
    if os.path.isdir(class_path):
        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        print(f"Class: {class_name} | Images: {len(image_files)}")

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),  # or your desired size
    transforms.ToTensor(),
])

dataset = ImageFolder(root="train", transform=transform)

# Feed it into a dataloader (batch size 8 here just for demo)
train_dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# View some examples
x, y = next(iter(train_dataloader))
print("Input shape:", x.shape)
print("Labels:", y)
plt.imshow(torchvision.utils.make_grid(x)[0], cmap="gray")

### U-Net

In [ ]:
class ClassConditionedUnet(nn.Module):
    def __init__(self, num_classes=4, class_emb_size=4):
        super().__init__()

        # The embedding layer will map the class label to a vector of size class_emb_size
        self.class_emb = nn.Embedding(num_classes, class_emb_size)

        # Self.model is an unconditional UNet with extra input channels to accept the conditioning information (the class embedding)
        self.model = UNet2DModel(
            sample_size=28,  # the target image resolution
            in_channels=1 + class_emb_size,  # Additional input channels for class cond.
            out_channels=1,  # the number of output channels
            layers_per_block=2,  # how many ResNet layers to use per UNet block
            block_out_channels=(32, 64, 64),
            down_block_types=(
                "DownBlock2D",  # a regular ResNet downsampling block
                "AttnDownBlock2D",  # a ResNet downsampling block with spatial self-attention
                "AttnDownBlock2D",
            ),
            up_block_types=(
                "AttnUpBlock2D",
                "AttnUpBlock2D",  # a ResNet upsampling block with spatial self-attention
                "UpBlock2D",  # a regular ResNet upsampling block
            ),
        )

    # Our forward method now takes the class labels as an additional argument
    def forward(self, x, t, class_labels):
        # Shape of x:
        bs, ch, w, h = x.shape

        # class conditioning in right shape to add as additional input channels
        class_cond = self.class_emb(class_labels)  # Map to embedding dimension
        class_cond = class_cond.view(bs, class_cond.shape[1], 1, 1).expand(bs, class_cond.shape[1], w, h)
        # x is shape (bs, 1, 28, 28) and class_cond is now (bs, 4, 28, 28)

        # Net input is now x and class cond concatenated together along dimension 1
        net_input = torch.cat((x, class_cond), 1)  # (bs, 5, 28, 28)

        # Feed this to the UNet alongside the timestep and return the prediction
        return self.model(net_input, t).sample  # (bs, 1, 28, 28)

In [ ]:
# Create a noise scheduler
noise_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")

### Training with checkpoints

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

# Assuming the dataset, ClassConditionedUnet, noise_scheduler, and device are already defined

train_dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
n_epochs = 100

net = ClassConditionedUnet(num_classes=4).to(device)
loss_fn = torch.nn.MSELoss()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
losses = []

checkpoint_path = "checkpoint_latest.pth"
start_epoch = 0

# Load from existing checkpoint if available
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    net.load_state_dict(checkpoint['model_state_dict'])
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    losses = checkpoint['losses']
    print(f"Resuming training from epoch {start_epoch}")
else:
    print("Starting training from scratch")

for epoch in range(start_epoch, n_epochs):
    for x, y in tqdm(train_dataloader):
        x = x.to(device) * 2 - 1
        y = y.to(device)
        noise = torch.randn_like(x)
        timesteps = torch.randint(0, 999, (x.shape[0],)).long().to(device)
        noisy_x = noise_scheduler.add_noise(x, noise, timesteps)

        pred = net(noisy_x, timesteps, y)
        loss = loss_fn(pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())

    avg_loss = sum(losses[-100:]) / 100
    print(f"Finished epoch {epoch}. Average of the last 100 loss values: {avg_loss:.6f}")

    # Save checkpoint after each epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': net.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'losses': losses,
    }, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

# Plot loss curve
plt.plot(losses)
plt.title("Training Loss Curve")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()


### Generation

Clear memory

In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


Image generation in separate class folders

In [ ]:

import os
import torch

# Path to your checkpoint file
checkpoint_path = "new_checkpoint.pth"   # <- change name here

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    net.load_state_dict(checkpoint['model_state_dict'])
    net.eval()
    print("Model loaded for generation from new_checkpoint.pth.")
else:
    raise FileNotFoundError("Checkpoint not found for generation.")

# Number of classes and images per class
num_classes = 4
images_per_class = 500
total_images = num_classes * images_per_class

# Generate samples
x = torch.randn(total_images, 1, 64, 64).to(device)
y = torch.tensor([[i] * images_per_class for i in range(num_classes)]).flatten().to(device)

for i, t in tqdm(enumerate(noise_scheduler.timesteps)):
    with torch.no_grad():
        residual = net(x, t, y)
    x = noise_scheduler.step(residual, t, x).prev_sample

# Save each image to its class folder
from torchvision.utils import save_image

x_cpu = x.detach().cpu().clip(-1, 1)
y_cpu = y.detach().cpu()

for idx, (img, label) in enumerate(zip(x_cpu, y_cpu)):
    folder = f"100eps_class_{label}"
    os.makedirs(folder, exist_ok=True)
    save_image((img + 1) / 2, os.path.join(folder, f"sample-round1_{idx}.png"))  # scale to [0,1]

print("Images saved to class folders.")

# Display 500 images for each class in a grid
import matplotlib.pyplot as plt
import torchvision

x_cpu = x.detach().cpu().clip(-1, 1)
y_cpu = y.detach().cpu()

for class_idx in range(num_classes):
    class_indices = (y_cpu == class_idx).nonzero(as_tuple=True)[0][:images_per_class]
    imgs = x_cpu[class_indices]
    # 500 images: 25 rows x 20 columns
    grid = torchvision.utils.make_grid(imgs, nrow=20, padding=2, normalize=True)
    plt.figure(figsize=(20, 25))
    plt.imshow(grid[0], cmap="gray")
    plt.title(f"Class {class_idx} - {images_per_class} generated images")
    plt.axis("off")
    plt.show()


## FID score

Installations

In [ ]:
%pip install torchmetrics
%pip install torch-fidelity

Resize all the input real images to make them of same size as generated images (64x64)

In [ ]:
import os
import random
from PIL import Image
from torchvision import transforms

input_root = "train"  # your original data folder
output_root = "train_resized_2000"  # new folder for sampled & resized images
target_size = (64, 64)
images_per_class = 2000

transform = transforms.Compose([
    transforms.Resize(target_size)
])

os.makedirs(output_root, exist_ok=True)

for class_name in os.listdir(input_root):
    class_input_path = os.path.join(input_root, class_name)
    class_output_path = os.path.join(output_root, class_name)
    os.makedirs(class_output_path, exist_ok=True)
    # List all image files in the class folder
    image_files = [f for f in os.listdir(class_input_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    # Randomly sample up to images_per_class
    sampled_files = random.sample(image_files, min(images_per_class, len(image_files)))
    for fname in sampled_files:
        img_path = os.path.join(class_input_path, fname)
        img = Image.open(img_path)
        img_resized = transform(img)
        img_resized.save(os.path.join(class_output_path, fname))

print("Sampled and resized images saved to", output_root)

### Add the new generated images to the generated_images folder after each round

Overall FID Score

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
import torch

# Prepare transforms and dataloaders
transform = transforms.Compose([
    transforms.Resize((299, 299)),  # InceptionV3 expects 299x299
    transforms.PILToTensor(),
])

real_dataset = ImageFolder('train_resized_2000', transform=transform)
gen_dataset = ImageFolder('generated_images', transform=transform)

real_loader = DataLoader(real_dataset, batch_size=32, shuffle=False)
gen_loader = DataLoader(gen_dataset, batch_size=32, shuffle=False)

fid = FrechetInceptionDistance(feature=2048).cuda()  # Use CUDA if available

# Add real images
for batch, _ in real_loader:
    fid.update(batch.cuda(), real=True)

# Add generated images
for batch, _ in gen_loader:
    fid.update(batch.cuda(), real=False)

score = fid.compute()
print(f"FID score: {score.item()}")

Per class FID Score

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
import os

transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.PILToTensor(),
])

real_root = 'train_sampled_resized'  # path to resized real images
gen_root = 'generated_images'

class_names = sorted(os.listdir(real_root))  # assumes subfolders per class

fid_scores = []

for class_name in class_names:
    real_dataset = ImageFolder(real_root, transform=transform)
    gen_dataset = ImageFolder(gen_root, transform=transform)

    # Filter indices for this class
    real_indices = [i for i, (_, label) in enumerate(real_dataset.samples) if real_dataset.classes[label] == class_name]
    gen_indices = [i for i, (_, label) in enumerate(gen_dataset.samples) if gen_dataset.classes[label] == class_name]

    # Subset datasets
    real_subset = torch.utils.data.Subset(real_dataset, real_indices)
    gen_subset = torch.utils.data.Subset(gen_dataset, gen_indices)

    # Skip if not enough samples for FID calculation
    if len(real_subset) < 2 or len(gen_subset) < 2:
        print(f"Skipping class {class_name}: not enough samples for FID calculation.")
        fid_scores.append(float('nan'))
        continue

    real_loader = DataLoader(real_subset, batch_size=32, shuffle=False)
    gen_loader = DataLoader(gen_subset, batch_size=32, shuffle=False)

    fid = FrechetInceptionDistance(feature=2048).cuda()

    for batch, _ in real_loader:
        fid.update(batch.cuda(), real=True)
    for batch, _ in gen_loader:
        fid.update(batch.cuda(), real=False)

    score = fid.compute().item()
    fid_scores.append(score)
    print(f"FID score for class {class_name}: {score}")

plt.figure(figsize=(8, 5))
plt.bar(class_names, fid_scores)
plt.xlabel("Class")
plt.ylabel("FID Score")
plt.title("FID Score per Class")
plt.savefig("fid_per_class.eps", format="eps", dpi=300, bbox_inches="tight")  # Save as high-quality EPS
plt.show()